In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# Set base directory in your Drive
base_dir = "/content/drive/MyDrive/NFATC4"

# Create necessary subfolders if they don't already exist
subdirs = [
    "sequence",         # FASTA, ESM embeddings
    "structure",        # AlphaFold PDB + coordinates
    "mutations",        # DMD-specific mutation flags
    "go_annotations",   # Gene Ontology TSV
    "rna_seq",          # RNA-seq BAM/VCF
    "ppi_networks",     # STRING edges
    "esm_embeddings",   # ESM-2 residue vectors
    "graph",            # Final graph representation
    "scripts"           # Utility scripts
]

for subdir in subdirs:
    os.makedirs(os.path.join(base_dir, subdir), exist_ok=True)

print("✅ Google Drive mounted and NFATC4 folders initialized.")


In [ ]:
import pandas as pd
import numpy as np

# ---- Define file paths ----
control_paths = [
    "/content/drive/MyDrive/dmd_dmd_graph/raw/quant/SRR16104020_quant/quant.sf",
    "/content/drive/MyDrive/dmd_dmd_graph/raw/quant/SRR3192550_quant/quant.sf",
    "/content/drive/MyDrive/dmd_dmd_graph/raw/quant/SRR7942570_quant/quant.sf"
]
dmd_paths = [
    "/content/drive/MyDrive/dmd_dmd_graph/raw/quant/SRR4240749_quant/quant.sf",
    "/content/drive/MyDrive/dmd_dmd_graph/raw/quant/SRR4240760_quant/quant.sf",
    "/content/drive/MyDrive/dmd_dmd_graph/raw/quant/SRR4240869_quant/quant.sf"
]

In [ ]:
# ---- Extract TPMs for NFATC4 ----
def extract_nfatc4_tpm(file_paths):
    tpms = []
    for path in file_paths:
        df = pd.read_csv(path, sep="\t")
        nfat = df[df["Name"].str.contains("NFATC4", case=False)]
        tpm = nfat["TPM"].sum() if not nfat.empty else 0
        tpms.append(tpm)
    return np.array(tpms)

control_tpms = extract_nfatc4_tpm(control_paths)
dmd_tpms     = extract_nfatc4_tpm(dmd_paths)

# ---- Compute average expression and fold change ----
mean_ctrl = control_tpms.mean()
mean_dmd  = dmd_tpms.mean()
log2_fc = np.log2((mean_dmd + 1e-5) / (mean_ctrl + 1e-5))

print("NFATC4 TPMs — Control:", control_tpms)
print("NFATC4 TPMs — DMD    :", dmd_tpms)
print(f"NFATC4 Mean TPM → Control: {mean_ctrl:.3f}, DMD: {mean_dmd:.3f}, log2 Fold Change: {log2_fc:.3f}")

In [ ]:
# Simulate mutation impact based on group log2 fold change
n_residues = 902
mutation_flags = np.zeros(n_residues)

# If strong downregulation, simulate domain disruption
if log2_fc < -1:  # More than 2-fold down in DMD
    sensitive_domains = [(40, 95), (200, 280), (700, 750)]  # From GO/domain knowledge
    for start, end in sensitive_domains:
        start = max(0, start)
        end = min(n_residues - 1, end)
        mutation_flags[start:end+1] = 1

np.save("/content/drive/MyDrive/NFATC4/mutations/simulated_dmd_mutation_vector.npy", mutation_flags)
print("✅ Updated mutation vector using all DMD and control samples.")


In [ ]:
import numpy as np

mutation_flags = np.load("/content/drive/MyDrive/NFATC4/mutations/simulated_dmd_mutation_vector.npy")

print(f"Shape: {mutation_flags.shape}")
print(f"Nonzero (mutated residues): {np.count_nonzero(mutation_flags)}")
print(f"Indices of mutations: {np.where(mutation_flags == 1)[0]}")


In [ ]:
import matplotlib.pyplot as plt

# Load mutation flags
mutation_flags = np.load("/content/drive/MyDrive/NFATC4/mutations/simulated_dmd_mutation_vector.npy")

# Define known domains
domains = {
    "DNA-binding": (40, 95),
    "Calcineurin-binding": (200, 280),
    "Transactivation": (700, 750)
}

plt.figure(figsize=(16, 2))
plt.title("Simulated DMD-Associated Disruption in NFATC4 by Residue Index", fontsize=14)
plt.plot(mutation_flags, lw=1.5, color="crimson", label="Disruption Flag")

# Add shaded domains
for label, (start, end) in domains.items():
    plt.axvspan(start, end, color='skyblue', alpha=0.3, label=label)

plt.xlabel("Residue Index (1-based)", fontsize=12)
plt.ylabel("Disruption", fontsize=12)
plt.ylim(-0.1, 1.1)
plt.legend(loc="upper right", fontsize=10)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/NFATC4/figures/nfatc4_mutation_domain_map.png", dpi=300)
plt.show()


In [ ]:
!pip install py3Dmol


In [ ]:
import py3Dmol
import numpy as np

# Load AlphaFold PDB file (chain A confirmed)
pdb_path = "/content/drive/MyDrive/NFATC4/structure/AF-Q14934-F1-model_v4.pdb"
with open(pdb_path, "r") as f:
    pdb_data = f.read()

# Load mutation vector
mutation_flags = np.load("/content/drive/MyDrive/NFATC4/mutations/simulated_dmd_mutation_vector.npy")
mutated_residues = np.where(mutation_flags == 1)[0] + 1  # 1-based for 3Dmol


In [ ]:
def show_structure(pdb_string, mutated_residues=None, title="NFATC4"):
    view = py3Dmol.view(width=600, height=500)
    view.addModel(pdb_string, "pdb")

    # Set default style
    view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'lightgrey'}})

    # Highlight mutated residues
    if mutated_residues:
        for res in mutated_residues:
            view.addStyle(
                {'chain': 'A', 'resi': str(res)},
                {'stick': {'color': 'red'}, 'sphere': {'color': 'red', 'radius': 0.5}}
            )

    view.setBackgroundColor('white')
    view.zoomTo()
    return view.show()


In [ ]:
show_structure(pdb_data, mutated_residues=None, title="NFATC4 Wild-Type")


In [ ]:
show_structure(pdb_data, mutated_residues=mutated_residues.tolist(), title="NFATC4 with DMD Disruptions")


In [ ]:
import requests

fasta_url = "https://rest.uniprot.org/uniprotkb/Q14934.fasta"
fasta_path = f"{base_dir}/sequence/Q14934.fasta"

r = requests.get(fasta_url)
with open(fasta_path, "wb") as f:
    f.write(r.content)

print("✅ Saved FASTA to:", fasta_path)


In [ ]:
af_url = "https://alphafold.ebi.ac.uk/files/AF-Q14934-F1-model_v4.pdb"
pdb_path = f"{base_dir}/structure/AF-Q14934-F1-model_v4.pdb"

r = requests.get(af_url)
with open(pdb_path, "wb") as f:
    f.write(r.content)

print("✅ Saved AlphaFold structure to:", pdb_path)


In [ ]:
!pip install fair-esm
!pip install biopython



In [ ]:
!wget "https://rest.uniprot.org/uniprotkb/Q14934.fasta" -O nfatc4.fasta


In [ ]:
import esm
import torch
from Bio import SeqIO
import numpy as np

# Load ESM-2 model
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()

# Read sequence
record = list(SeqIO.parse("nfatc4.fasta", "fasta"))[0]
sequence = str(record.seq)
data = [("NFATC4", sequence)]
batch_labels, batch_strs, batch_tokens = batch_converter(data)

# Disable gradients for inference
with torch.no_grad():
    results = model(batch_tokens, repr_layers=[33], return_contacts=False)

# Extract per-residue embeddings (layer 33, remove BOS/EOS)
token_representations = results["representations"][33]
embedding = token_representations[0, 1:-1]  # (921, 1280)

# Save
np.save("/content/drive/MyDrive/NFATC4/esm_embeddings/Q14934_esm.npy", embedding.cpu().numpy())
print("✅ Saved full-length ESM-2 embedding:", embedding.shape)


In [ ]:
import requests

go_url = "https://www.ebi.ac.uk/QuickGO/services/annotation/downloadSearch"
params = {
    "geneProductId": "UniProtKB:Q14934",
    "aspect": "biological_process,molecular_function,cellular_component",
    "format": "tsv"
}
headers = {
    "Accept": "text/tsv"
}

# Save to your base_dir
go_path = f"{base_dir}/go_annotations/Q14934_go.tsv"

# Download with correct headers
response = requests.get(go_url, params=params, headers=headers)

# Save if successful
if response.status_code == 200:
    with open(go_path, "w") as f:
        f.write(response.text)
    print("✅ GO annotations saved to:", go_path)
else:
    print("❌ Failed:", response.status_code)
    print("Response:", response.text[:300])


In [ ]:
!pip install torch_geometric

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import torch
from torch_geometric.data import Data
from Bio.PDB import PDBParser
from scipy.spatial import distance_matrix

# Paths
base_dir = "/content/drive/MyDrive/NFATC4"
pdb_path = f"{base_dir}/structure/AF-Q14934-F1-model_v4.pdb"
esm_path = f"{base_dir}/esm_embeddings/Q14934_esm.npy"
mut_path = f"{base_dir}/mutations/simulated_dmd_mutation_vector.npy"

# Load features
esm_embeddings = np.load(esm_path)                  # (902, 1280)
mutation_flags = np.load(mut_path).reshape(-1, 1)   # (902, 1)

assert esm_embeddings.shape[0] == mutation_flags.shape[0] == 902

# Load AlphaFold structure and extract Cα coordinates
parser = PDBParser(QUIET=True)
structure = parser.get_structure("NFATC4", pdb_path)
chain = structure[0]["A"]

coords = []
for res in chain:
    if "CA" in res:
        coords.append(res["CA"].get_coord())
coords = np.array(coords)
pos = torch.tensor(coords, dtype=torch.float32)  # (902, 3)

# Edge construction: residues < 8Å apart (exclude self-edges)
dist = distance_matrix(coords, coords)
i, j = np.where((dist < 8.0) & (dist > 0.0))
edge_index = torch.tensor([i, j], dtype=torch.long)  # shape (2, num_edges)

# Combine ESM and mutation info
esm_tensor = torch.tensor(esm_embeddings, dtype=torch.float32)  # (902, 1280)
mut_tensor = torch.tensor(mutation_flags, dtype=torch.float32)  # (902, 1)
node_features = torch.cat([esm_tensor, mut_tensor], dim=1)      # (902, 1281)

# Build PyG Data object
data = Data(
    x=node_features,
    edge_index=edge_index,
    pos=pos,
    name="NFATC4"
)

# Save the graph
save_path = f"{base_dir}/graph/nfatc4_graph.pt"
torch.save(data, save_path)
print("✅ Graph built and saved to:", save_path)
print("📐 Nodes:", data.num_nodes, "| 🧩 Edges:", data.edge_index.shape[1])


In [ ]:
!pip install biopython torch_geometric matplotlib


In [ ]:
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Load saved graph
import torch

data = torch.load("/content/drive/MyDrive/NFATC4/graph/nfatc4_graph.pt", weights_only=False)


pos = data.pos.numpy()               # (902, 3)
mut_flags = data.x[:, -1].numpy()    # last dim = mutation flag (0 or 1)
edges = data.edge_index.numpy()      # (2, E)

# Split into x, y, z
x, y, z = pos[:, 0], pos[:, 1], pos[:, 2]

# Start 3D plot
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection="3d")
ax.set_title("NFATC4 Protein Graph (DMD Mutation-Aware)", fontsize=14)

# Plot edges (protein graph structure)
for i in range(edges.shape[1]):
    u, v = edges[:, i]
    ax.plot([x[u], x[v]], [y[u], y[v]], [z[u], z[v]], c='gray', alpha=0.2)

# Plot residues: mutated in red, normal in blue
colors = ['red' if flag == 1 else 'dodgerblue' for flag in mut_flags]
ax.scatter(x, y, z, c=colors, s=20)

# Legend
from matplotlib.lines import Line2D
legend = [
    Line2D([0], [0], marker='o', color='w', label='Mutated Residue', markerfacecolor='red', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Normal Residue', markerfacecolor='dodgerblue', markersize=10),
]
ax.legend(handles=legend, loc='upper left')

# Axes labels
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

n = data.num_nodes
mutation_flags = data.x[:, -1].numpy().astype(int)

# ---------------------
# Functional domain labels
# ---------------------
domain_regions = {
    "Transactivation A": (1, 100),
    "Calcineurin-binding": (110, 140),
    "DNA-binding (RHR)": (140, 260),
    "Regulatory": (260, 400),
    "Transactivation B": (650, 900),
}

# ---------------------
# Mutation statistics by domain
# ---------------------
print("🧬 NFATC4 Domain-Level Mutation Summary:\n")
total_mutations = 0

for name, (start, end) in domain_regions.items():
    start_idx = max(0, start)
    end_idx = min(n, end + 1)
    domain_mutations = np.sum(mutation_flags[start_idx:end_idx])
    domain_length = end_idx - start_idx
    mutation_rate = domain_mutations / domain_length

    print(f"📌 Domain: {name}")
    print(f"   Residues: {start_idx}–{end_idx - 1}")
    print(f"   Length: {domain_length}")
    print(f"   Mutated Residues: {domain_mutations}")
    print(f"   Mutation Rate: {mutation_rate:.2%}")

    # Interpretation
    if domain_mutations > 0:
        if "DNA-binding" in name:
            print("   🔬 Effect: Mutations in DNA-binding domain may disrupt NFATC4's ability to regulate gene expression.")
        elif "Calcineurin" in name:
            print("   ⚠️ Effect: Mutations in this domain could impair calcium/calcineurin-mediated activation signaling.")
        elif "Regulatory" in name:
            print("   🧩 Effect: Could disturb autoregulation, leading to aberrant nuclear translocation or activity.")
        elif "Transactivation" in name:
            print("   📉 Effect: May impact NFATC4’s ability to activate downstream transcriptional programs.")
    else:
        print("   ✅ No mutations detected in this domain.")

    print()
    total_mutations += domain_mutations

print(f"🔢 Total Mutated Residues Across NFATC4: {total_mutations} / {n}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing

# --- GVP Block ---
class GVP(nn.Module):
    def __init__(self, in_dim, out_dim, act=F.gelu):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
            nn.Linear(out_dim, out_dim),
            nn.LayerNorm(out_dim)
        )
        self.act = act

    def forward(self, x):
        return self.act(self.proj(x))

# --- Multi-head GVP Conv ---
class MultiHeadGVPConv(MessagePassing):
    def __init__(self, in_dim, out_dim, heads=4, dropout=0.25):
        super().__init__(aggr='add')
        self.heads = heads
        self.dropout = nn.Dropout(dropout)

        self.msg_mlps = nn.ModuleList([GVP(in_dim, out_dim) for _ in range(heads)])
        self.agg_norm = nn.LayerNorm(out_dim)
        self.residual = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x, edge_index):
        return self.propagate(edge_index, x=x)

    def message(self, x_j):
        messages = [mlp(x_j) for mlp in self.msg_mlps]
        msg_out = torch.stack(messages).mean(dim=0)
        return self.dropout(msg_out)

    def update(self, aggr_out, x):
        return self.agg_norm(aggr_out + self.residual(x))

# --- Final Model ---
class BayesianGVP_GNN(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, dropout=0.3, heads=4):
        super().__init__()
        self.input_norm = nn.LayerNorm(in_dim)

        self.block1 = MultiHeadGVPConv(in_dim, hidden_dim, heads, dropout)
        self.block2 = MultiHeadGVPConv(hidden_dim, hidden_dim, heads, dropout)
        self.block3 = MultiHeadGVPConv(hidden_dim, hidden_dim, heads, dropout)

        self.dropout = nn.Dropout(dropout)
        self.final_norm = nn.LayerNorm(hidden_dim)

        # Deep classifier head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x, edge_index):
        x = self.input_norm(x)
        x = self.block1(x, edge_index)
        x = F.gelu(x)
        x = self.dropout(x)

        x = self.block2(x, edge_index)
        x = F.gelu(x)
        x = self.dropout(x)

        x = self.block3(x, edge_index)
        x = F.gelu(self.final_norm(x))
        x = self.dropout(x)

        return self.classifier(x).view(-1)


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data

# Load saved graph
data = torch.load("/content/drive/MyDrive/NFATC4/graph/nfatc4_graph.pt", weights_only=False)

# Extract all 1281 features (ESM + mutation flag)
x = data.x                 # (902, 1281)
y = data.x[:, -1]          # Mutation labels (still binary 0 or 1)

# Train/val split (node-level)
num_nodes = data.num_nodes
perm = torch.randperm(num_nodes)
train_idx = perm[:int(0.8 * num_nodes)]
val_idx = perm[int(0.8 * num_nodes):]

# Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BayesianGVP_GNN(in_dim=x.shape[1], hidden_dim=256, dropout=0.3, heads=4).to(device)
data = data.to(device)
x, y = x.to(device), y.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.004, weight_decay=1e-5)

# --- Training Loop ---
for epoch in range(1, 101):
    model.train()
    optimizer.zero_grad()
    logits = model(x, data.edge_index)
    loss = F.binary_cross_entropy_with_logits(logits[train_idx], y[train_idx])
    loss.backward()
    optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        val_logits = model(x, data.edge_index)
        val_probs = torch.sigmoid(val_logits[val_idx])
        val_preds = (val_probs > 0.5).float()
        val_acc = (val_preds == y[val_idx]).float().mean().item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f} | Val Acc: {val_acc:.4f}")


In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/NFATC4/bayesian_gvp_model.pt")


In [ ]:
# ----------------------
# Load graph and model
# ----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the graph
data = torch.load("/content/drive/MyDrive/NFATC4/graph/nfatc4_graph.pt", weights_only=False)
x = data.x.to(device)
y = data.x[:, -1].to(device)
edge_index = data.edge_index.to(device)

# Instantiate model
model = BayesianGVP_GNN(in_dim=x.shape[1]).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/NFATC4/bayesian_gvp_model.pt"))  # use correct weights file
model.eval()

# ----------------------
# MC-Dropout Prediction
# ----------------------
def mc_dropout_predict(model, x, edge_index, mc_samples=50):
    model.train()
    preds = []
    for _ in range(mc_samples):
        with torch.no_grad():
            logits = model(x, edge_index)
            probs = torch.sigmoid(logits)
            preds.append(probs.cpu().numpy())
    preds = np.stack(preds)
    return preds.mean(axis=0), preds.std(axis=0)

# ----------------------
# Domain AUC
# ----------------------
domain_regions = {
    "Transactivation A": (1, 100),
    "Calcineurin-binding": (110, 140),
    "DNA-binding (RHR)": (140, 260),
    "Regulatory": (260, 400),
    "Transactivation B": (650, 900),
}

def per_domain_metrics(preds, labels, domain_regions):
    results = []
    for name, (start, end) in domain_regions.items():
        dom_labels = labels[start:end+1]
        dom_preds = preds[start:end+1]
        if np.any(dom_labels == 1):
            auc_score = roc_auc_score(dom_labels, dom_preds)
        else:
            auc_score = float('nan')
        results.append((name, end - start + 1, int(dom_labels.sum()), auc_score))
    return results

# ----------------------
# PR Curve
# ----------------------
def pr_curve_stats(preds, labels):
    precision, recall, _ = precision_recall_curve(labels, preds)
    pr_auc = auc(recall, precision)
    return precision, recall, pr_auc

def plot_pr_curve(precision, recall, pr_auc):
    plt.figure(figsize=(6, 4))
    plt.plot(recall, precision, label=f'PR AUC = {pr_auc:.3f}', color='blue')
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.legend()
    plt.grid(True)
    plt.show()

# ----------------------
# Run All Metrics
# ----------------------
mean_pred, std_pred = mc_dropout_predict(model, x, edge_index, mc_samples=50)
labels_np = y.cpu().numpy()

# Domain metrics
domain_results = per_domain_metrics(mean_pred, labels_np, domain_regions)
print("📊 Domain-wise AUC Results:")
for name, length, pos, auc_score in domain_results:
    print(f"• {name:<22} | Len: {length:3d} | Mutated: {pos:3d} | AUC: {auc_score:.4f}")

# PR Curve
precision, recall, pr_auc = pr_curve_stats(mean_pred, labels_np)
plot_pr_curve(precision, recall, pr_auc)
print(f"🧬 Overall PR AUC: {pr_auc:.4f}")

In [ ]:
def compute_saliency(model, x, edge_index, target_idx=None):
    x.requires_grad = True
    model.eval()

    out = model(x, edge_index)
    target = out if target_idx is None else out[target_idx]
    target.backward(torch.ones_like(target))

    saliency = x.grad.abs().sum(dim=1).detach().cpu().numpy()  # (902,)
    return saliency


In [ ]:
def get_activation_norms(model, x, edge_index):
    model.eval()
    with torch.no_grad():
        x1 = model.input_norm(x)
        h1 = model.block1(x1, edge_index)
        h2 = model.block2(h1, edge_index)
        h3 = model.block3(h2, edge_index)
        return h3.norm(dim=1).cpu().numpy()  # (902,)


In [ ]:
from scipy.stats import spearmanr

def correlate_saliency_expression(saliency, delta_expr):
    rho, p = spearmanr(saliency, delta_expr)
    print(f"📈 Spearman r = {rho:.3f}, p = {p:.3e}")
    return rho, p


In [ ]:
import matplotlib.pyplot as plt

def plot_residue_signal(signal, title="Saliency"):
    plt.figure(figsize=(12, 3))
    plt.plot(signal, color='crimson')
    plt.xlabel("Residue Index")
    plt.ylabel("Signal")
    plt.title(title)
    plt.grid(True)
    plt.show()


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import precision_recall_curve, auc

# Load graph and model (ensure you've run your model definition first)
graph_path = "/content/drive/MyDrive/NFATC4/graph/nfatc4_graph.pt"
model_path = "/content/drive/MyDrive/NFATC4/bayesian_gvp_model.pt"

# Load graph
data = torch.load(graph_path, weights_only=False)
x = data.x
y = data.x[:, -1]
edge_index = data.edge_index

# Load model
model = BayesianGVP_GNN(in_dim=x.shape[1])
model.load_state_dict(torch.load(model_path, map_location=torch.device("cpu")))
model.eval()

# Move data to CPU to reduce Colab RAM usage
x.requires_grad = True
logits = model(x, edge_index)
logits.sum().backward()  # full backprop

# Residue-level saliency
saliency = x.grad.abs().sum(dim=1).detach().cpu().numpy()

# Simulated delta expression vector (replace this with real values later)
np.random.seed(0)
delta_expr = np.random.randn(len(saliency))

# Correlation (Spearman)
rho, pval = spearmanr(saliency, delta_expr)
print(f"🔬 Spearman Correlation between saliency and expression change: ρ = {rho:.4f}, p = {pval:.4e}")

# Plot saliency
plt.figure(figsize=(12, 4))
plt.plot(saliency, color='darkred', label="Saliency")
plt.title("Residue-Level Saliency Map (Bayesian GVP-GNN)")
plt.xlabel("Residue Index")
plt.ylabel("Saliency Score")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Saliency: how much each residue influences mutation prediction
x.requires_grad = True
logits = model(x, edge_index)
logits.sum().backward()
saliency = x.grad.abs().sum(dim=1).detach().cpu().numpy()
np.save("/content/drive/MyDrive/NFATC4/saliency_vector.npy", saliency)
print("✅ Saved saliency_vector.npy")


In [ ]:
def mc_dropout_predict(model, x, edge_index, mc_samples=50):
    model.train()
    preds = []
    for _ in range(mc_samples):
        with torch.no_grad():
            logits = model(x, edge_index)
            probs = torch.sigmoid(logits)
            preds.append(probs.cpu().numpy())
    preds = np.stack(preds)
    return preds.mean(axis=0), preds.std(axis=0)

mean_pred, std_pred = mc_dropout_predict(model, x, edge_index, mc_samples=50)
np.save("/content/drive/MyDrive/NFATC4/std_pred_vector.npy", std_pred)
print("✅ Saved std_pred_vector.npy")


In [ ]:
x_ctrl = x.clone()
x_dmd = x.clone()
x_dmd[:, -1] = 1  # simulate DMD mutation on final feature

# Run forward passes
h_ctrl = model.block3(model.block2(model.block1(x_ctrl, edge_index), edge_index), edge_index)
h_dmd = model.block3(model.block2(model.block1(x_dmd, edge_index), edge_index), edge_index)

delta_hidden = (h_dmd - h_ctrl).norm(dim=1).detach().cpu().numpy()
np.save("/content/drive/MyDrive/NFATC4/delta_hidden_vector.npy", delta_hidden)
print("✅ Saved delta_hidden_vector.npy")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import label, gaussian_filter1d
import pandas as pd

# ----------------------------
# Load previously saved arrays
# ----------------------------
saliency = np.load("/content/drive/MyDrive/NFATC4/saliency_vector.npy")
std_pred = np.load("/content/drive/MyDrive/NFATC4/std_pred_vector.npy")
delta_hidden = np.load("/content/drive/MyDrive/NFATC4/delta_hidden_vector.npy")

# ----------------------------
# Compute & Smooth Joint Score
# ----------------------------
joint_score_raw = saliency * std_pred * delta_hidden
joint_score = gaussian_filter1d(joint_score_raw, sigma=2)  # smoothing
top_indices = np.argsort(-joint_score)[:30]

# ----------------------------
# Threshold + Cluster Detection
# ----------------------------
threshold = np.percentile(joint_score, 93)  # slightly more inclusive
binary_map = (joint_score > threshold).astype(int)
clusters, _ = label(binary_map)

# ----------------------------
# Create per-residue DataFrame
# ----------------------------
df = pd.DataFrame({
    "Residue": np.arange(len(joint_score)),
    "Saliency": saliency,
    "Uncertainty": std_pred,
    "DeltaHidden": delta_hidden,
    "JointScore": joint_score,
    "ClusterID": clusters
})
df_top = df[df["ClusterID"] > 0].sort_values("JointScore", ascending=False).head(30)

# ----------------------------
# Plot Smoothed Joint Score
# ----------------------------
plt.figure(figsize=(14, 4))
plt.plot(joint_score_raw, alpha=0.4, label="Raw Joint Score", color="lightgray")
plt.plot(joint_score, label="Smoothed Joint Score", color="darkblue")
plt.scatter(top_indices, joint_score[top_indices], color='crimson', label="Top Residues")
plt.title("Smoothed Joint Score: Saliency × Uncertainty × ΔRepresentation")
plt.xlabel("Residue Index")
plt.ylabel("Joint Score")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# ----------------------------
# Print top residue-level hits
# ----------------------------
print("🧬 Top 30 Residue Candidates (Functional, Uncertain, DMD-sensitive):")
print(df_top.to_string(index=False))

# ----------------------------
# Identify section-level cluster ranges
# ----------------------------
cluster_ranges = []
min_len = 5  # only keep clusters ≥5 residues long
for cluster_id in np.unique(clusters):
    if cluster_id == 0: continue
    positions = np.where(clusters == cluster_id)[0]
    if len(positions) < min_len:
        continue
    start, end = positions[0], positions[-1]
    avg_score = joint_score[positions].mean()
    max_score = joint_score[positions].max()
    cluster_ranges.append({
        "ClusterID": cluster_id,
        "Start": start,
        "End": end,
        "Length": end - start + 1,
        "AvgJointScore": avg_score,
        "MaxJointScore": max_score
    })

df_clusters = pd.DataFrame(cluster_ranges).sort_values("AvgJointScore", ascending=False)

# ----------------------------
# Print residue *section* targets
# ----------------------------
print("\n📍 High-Impact Residue Sections (Smoothed & Filtered):")
print(df_clusters.to_string(index=False))


In [ ]:
import py3Dmol
from google.colab import drive
drive.mount('/content/drive')

# Load PDB file as string
with open("/content/drive/MyDrive/NFATC4/structure/AF-Q14934-F1-model_v4.pdb", "r") as f:
    pdb_data = f.read()

# Highlight clusters from your `df_clusters`
highlight_ranges = [
    (row['Start'], row['End']) for _, row in df_clusters.iterrows()
]

# Initialize viewer
view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_data, 'pdb')
view.setStyle({'cartoon': {'color': 'lightgrey'}})

# Highlight important clusters in red
for start, end in highlight_ranges:
    start = int(start)
    end = int(end)
    view.addStyle(
        {'resi': list(range(start + 1, end + 2))},  # PDB is 1-indexed
        {'cartoon': {'color': 'red'}, 'stick': {}}
    )

view.zoomTo()
view.show()


In [ ]:
!apt-get install -y dssp
!pip install biopython


In [ ]:
from Bio.PDB import PDBParser, DSSP
import numpy as np

# Load PDB structure
pdb_path = "/content/drive/MyDrive/NFATC4/structure/AF-Q14934-F1-model_v4.pdb"
parser = PDBParser()
structure = parser.get_structure("NFATC4", pdb_path)

# Run DSSP
model = structure[0]
dssp = DSSP(model, pdb_path)

# Extract RSA per residue
rsa = []
for key in dssp.keys():
    rsa_value = dssp[key][3]  # Relative accessibility (0–1)
    rsa.append(rsa_value)

rsa_array = np.array(rsa)
np.save("/content/drive/MyDrive/NFATC4/rsa_vector.npy", rsa_array)

print("✅ Saved rsa_vector.npy with shape:", rsa_array.shape)


In [ ]:
import numpy as np
import pandas as pd

# Load cluster summary (from earlier step)
# df_clusters = your previously computed DataFrame
# Already contains: Start, End, Length, AvgJointScore, MaxJointScore

# Load RSA (0–1 per residue) and domain annotations
rsa = np.load("/content/drive/MyDrive/NFATC4/rsa_vector.npy")

# Optional: load known domain boundaries for filtering/visual context
canonical_domains = [
    (1, 100),    # Transactivation A
    (110, 140),  # Calcineurin-binding
    (140, 260),  # DNA-binding
    (260, 400),  # Regulatory
    (650, 900)   # Transactivation B
]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import label, gaussian_filter1d
import pandas as pd

# ----------------------------
# Load previously saved arrays
# ----------------------------
saliency = np.load("/content/drive/MyDrive/NFATC4/saliency_vector.npy")
std_pred = np.load("/content/drive/MyDrive/NFATC4/std_pred_vector.npy")
delta_hidden = np.load("/content/drive/MyDrive/NFATC4/delta_hidden_vector.npy")
rsa = np.load("/content/drive/MyDrive/NFATC4/rsa_vector.npy")

# ----------------------------
# Define canonical functional domains
# ----------------------------
canonical_domains = [
    (1, 100),    # Transactivation A
    (110, 140),  # Calcineurin-binding
    (140, 260),  # DNA-binding
    (260, 400),  # Regulatory
    (650, 900)   # Transactivation B
]

# ----------------------------
# Compute & Smooth Joint Score
# ----------------------------
joint_score_raw = saliency * std_pred * delta_hidden
joint_score = gaussian_filter1d(joint_score_raw, sigma=2)  # smoothing
top_indices = np.argsort(-joint_score)[:30]

# ----------------------------
# Threshold + Cluster Detection
# ----------------------------
threshold = np.percentile(joint_score, 93)  # slightly more inclusive
binary_map = (joint_score > threshold).astype(int)
clusters, _ = label(binary_map)

# ----------------------------
# Cluster Range Extraction
# ----------------------------
cluster_ranges = []
min_len = 5  # filter out short clusters
for cluster_id in np.unique(clusters):
    if cluster_id == 0:
        continue
    indices = np.where(clusters == cluster_id)[0]
    if len(indices) < min_len:
        continue
    start, end = indices[0], indices[-1]
    avg_score = joint_score[indices].mean()
    max_score = joint_score[indices].max()
    cluster_ranges.append({
        "ClusterID": cluster_id,
        "Start": start,
        "End": end,
        "Length": end - start + 1,
        "AvgJointScore": avg_score,
        "MaxJointScore": max_score
    })

df_clusters = pd.DataFrame(cluster_ranges).sort_values("AvgJointScore", ascending=False)

# ----------------------------
# Compute Avg RSA + Distance to Domains
# ----------------------------
def distance_to_domains(start, end, domains):
    distances = []
    for dom_start, dom_end in domains:
        if end < dom_start:
            distances.append(dom_start - end)
        elif start > dom_end:
            distances.append(start - dom_end)
        else:
            distances.append(0)  # overlaps domain
    return min(distances)

rsa_avg = []
domain_dist = []

for _, row in df_clusters.iterrows():
    idxs = np.arange(int(row.Start), int(row.End)+1)
    rsa_avg.append(rsa[idxs].mean())
    domain_dist.append(distance_to_domains(int(row.Start), int(row.End), canonical_domains))

df_clusters["AvgRSA"] = rsa_avg
df_clusters["DistToDomain"] = domain_dist

# ----------------------------
# Print Final Prioritized Clusters
# ----------------------------
print("📍 High-Impact Residue Sections (Smoothed, with RSA and Domain Distance):")
print(df_clusters.to_string(index=False))


In [ ]:
!pip install -q nglview biopython

In [ ]:
!pip install -q py3Dmol

import py3Dmol

# Load AlphaFold structure
pdb_path = "/content/drive/MyDrive/NFATC4/structure/AF-Q14934-F1-model_v4.pdb"
with open(pdb_path, 'r') as f:
    pdb_data = f.read()

# Set up viewer
view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_data, 'pdb')
view.setStyle({'cartoon': {'color': 'lightgrey'}})

# Highlight clusters manually
highlight_clusters = {
    "Cluster1_36_44": list(range(36, 45)),
    "Cluster2_88_99": list(range(88, 100)),
    "Cluster3_104_115": list(range(104, 116)),
    "Cluster4_195_203": list(range(195, 204)),
    "Cluster5_278_283": list(range(278, 284)),
    "Cluster6_696_704": list(range(696, 705)),
    "Cluster7_747_753": list(range(747, 754)),
}

colors = ["red", "orange", "yellow", "green", "cyan", "blue", "magenta"]

for i, (name, residues) in enumerate(highlight_clusters.items()):
    for r in residues:
        view.addStyle({'resi': str(r)}, {'stick': {'color': colors[i]}, 'sphere': {'color': colors[i]}})

view.zoomTo()
view.show()


In [ ]:
import pandas as pd

# --- Your cluster list (from earlier)
clusters = [
    {"ClusterID": 1, "Start": 36, "End": 44},
    {"ClusterID": 2, "Start": 88, "End": 99},
    {"ClusterID": 3, "Start": 104, "End": 115},
    {"ClusterID": 4, "Start": 195, "End": 203},
    {"ClusterID": 5, "Start": 278, "End": 283},
    {"ClusterID": 6, "Start": 696, "End": 704},
    {"ClusterID": 7, "Start": 747, "End": 753},
]

# --- Manually curated NFATC4 functional domains (from UniProt Q14934)
domains = [
    {"Domain": "Transactivation Domain (TAD)", "Start": 1, "End": 300},
    {"Domain": "Calcineurin Docking (PxIxIT)", "Start": 250, "End": 270},
    {"Domain": "Nuclear Localization Signal (NLS)", "Start": 265, "End": 290},
    {"Domain": "Rel Homology DNA-binding (RHD)", "Start": 330, "End": 600},
    {"Domain": "Nuclear Export Signal (NES)", "Start": 610, "End": 650},
]

# --- Function to check if two ranges overlap
def overlaps(start1, end1, start2, end2):
    return max(start1, start2) <= min(end1, end2)

# --- Annotate each cluster with overlapping domain(s)
annotated_clusters = []

for cluster in clusters:
    matching_domains = []
    for domain in domains:
        if overlaps(cluster["Start"], cluster["End"], domain["Start"], domain["End"]):
            matching_domains.append(domain["Domain"])
    annotated_clusters.append({
        "ClusterID": cluster["ClusterID"],
        "Start": cluster["Start"],
        "End": cluster["End"],
        "Length": cluster["End"] - cluster["Start"] + 1,
        "Overlapping Domains": ", ".join(matching_domains) if matching_domains else "None"
    })

# --- Convert to DataFrame and display
df_annotated = pd.DataFrame(annotated_clusters)
print(df_annotated.to_string(index=False))


In [ ]:
!apt-get install -y build-essential git cmake libglew-dev
!git clone https://github.com/Discngine/fpocket.git
!cd fpocket && make
!pip install prody biopython


In [ ]:
# Copy your AlphaFold NFATC4 PDB file
!cp "/content/drive/MyDrive/NFATC4/structure/AF-Q14934-F1-model_v4.pdb" /content/nfatc4.pdb

# Run fpocket
!./fpocket/bin/fpocket -f /content/nfatc4.pdb


In [ ]:
!pip install scipy==1.9.3 --force-reinstall


In [ ]:
from prody import *
from pylab import *

# Parse PDB and build model
structure = parsePDB("/content/nfatc4.pdb")
calphas = structure.select('protein and name CA')

# Build elastic network and calculate modes
anm = ANM('NFATC4_ANM')
anm.buildHessian(calphas)
anm.calcModes(n_modes=20)

# Calculate dynamic cross-correlation (top 6 modes)
cc = calcCrossCorr(anm[:6], n_cpu=1)

import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(10, 8))
plt.imshow(cc, cmap='bwr', interpolation='nearest', vmin=-1, vmax=1)
plt.colorbar(label="Cross-Correlation")
plt.title("NFATC4 Dynamic Cross-Correlation (Modes 1–6)")
plt.xlabel("Residue Index")
plt.ylabel("Residue Index")
plt.tight_layout()
plt.show()






In [ ]:
import os
import pandas as pd  # ✅ ADD THIS

# Step: extract which residues are in which pockets
allosteric_candidates = []

for pocket_id in range(1, 11):  # check top 10 pockets
    pocket_file = f"/content/nfatc4_out/pockets/pocket{pocket_id}_atm.pdb"
    if not os.path.exists(pocket_file): continue
    res_ids = []
    with open(pocket_file) as f:
        for line in f:
            if line.startswith("ATOM"):
                try:
                    resnum = int(line[22:26].strip())
                    res_ids.append(resnum)
                except: continue
    if not res_ids: continue
    # Compute average coupling of this pocket to the DNA-binding domain (res 330–600)
    mean_corr_to_domain = cc[:, res_ids].mean(axis=1)[330:600].mean()
    allosteric_candidates.append({
        "Pocket": pocket_id,
        "MeanCorrelationToDNA": mean_corr_to_domain,
        "Residues": sorted(set(res_ids))
    })

# Rank pockets by dynamic coupling to RHD domain
df_allosteric = pd.DataFrame(allosteric_candidates)
df_allosteric = df_allosteric.sort_values("MeanCorrelationToDNA", ascending=False)
print(df_allosteric)


In [ ]:
import numpy as np
from prody import *

# Load structure and C-alpha atoms
structure = parsePDB("/content/nfatc4.pdb")
calphas = structure.select('protein and name CA')
n_residues = len(calphas)

# Load simulated mutation vector
mutation_flags = np.load("/content/drive/MyDrive/NFATC4/mutations/simulated_dmd_mutation_vector.npy")

# Build baseline ANM and extract Hessian
anm_mut = ANM('Mutated_NFATC4_ANM')
anm_mut.buildHessian(calphas, gamma=1.0)
H = anm_mut.getHessian().copy()  # extract a copy of the Hessian matrix

# Damp connections for mutated residues
for i in range(n_residues):
    if mutation_flags[i] == 1:
        H[i*3:(i+1)*3, :] *= 0.2  # soften row
        H[:, i*3:(i+1)*3] *= 0.2  # soften col

# Set the modified Hessian back into the model
anm_mut.setHessian(H)
anm_mut.calcModes(n_modes=20)

# Calculate correlation matrix for mutant
cc_mut = calcCrossCorr(anm_mut[:6], n_cpu=1)


In [ ]:
# Use same RHD domain: residues 330–600
delta_allosteric = []

for pocket in allosteric_candidates:
    res_ids = pocket["Residues"]
    if not res_ids: continue

    wt_coupling = cc[:, res_ids].mean(axis=1)[330:600].mean()
    mut_coupling = cc_mut[:, res_ids].mean(axis=1)[330:600].mean()

    delta_allosteric.append({
        "Pocket": pocket["Pocket"],
        "WT_Coupling": wt_coupling,
        "Mutant_Coupling": mut_coupling,
        "DeltaCoupling": mut_coupling - wt_coupling,
        "Residues": res_ids
    })

import pandas as pd
df_delta = pd.DataFrame(delta_allosteric).sort_values("DeltaCoupling", ascending=False)
print(df_delta)


In [ ]:
# Filter pockets that are only significant in the mutant
df_mutant_only = df_delta[
    (df_delta['WT_Coupling'] < 0.5) &
    (df_delta['Mutant_Coupling'] > 0.6) &
    (df_delta['DeltaCoupling'] > 0.1)
].copy()

print("🧬 Mutation-specific allosteric pocket candidates:")
print(df_mutant_only)


In [ ]:
!pip install -q py3Dmol

import py3Dmol
import numpy as np

# Load structure
with open("/content/nfatc4.pdb", 'r') as f:
    pdb_data = f.read()

# Load DMD mutation clusters
mutation_clusters = np.where(np.load("/content/drive/MyDrive/NFATC4/mutations/simulated_dmd_mutation_vector.npy") == 1)[0].tolist()

# RHD DNA-binding domain
rhd_domain = list(range(330, 601))

# Get top mutation-specific pocket
top_mut_only = df_mutant_only.iloc[0]
top_mut_residues = top_mut_only['Residues']

# Set up viewer
view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_data, "pdb")
view.setStyle({'cartoon': {'color': 'lightgrey'}})

# Highlight top mutation-only allosteric pocket in GREEN
for res in top_mut_residues:
    view.addStyle({'resi': str(res)}, {'stick': {'color': 'green'}, 'sphere': {'color': 'green'}})

# Highlight mutation clusters in RED
for res in mutation_clusters:
    view.addStyle({'resi': str(res)}, {'stick': {'color': 'red'}, 'sphere': {'color': 'red'}})

# Highlight RHD domain in BLUE
for res in rhd_domain:
    view.addStyle({'resi': str(res)}, {'cartoon': {'color': 'blue'}})

view.zoomTo()
view.show()


In [ ]:
# Your clusters
clusters = {
    1: list(range(36, 45)),
    2: list(range(88, 100)),
    3: list(range(104, 116)),
    4: list(range(195, 204)),
    5: list(range(278, 284)),
    6: list(range(696, 705)),
    7: list(range(747, 754))
}

# Mutation-specific allosteric pockets
pockets = {
    10: [434, 435, 527, 528, 529, 530, 533],
    7:  [478, 479, 480, 483, 502, 503, 504, 529, 537],
    9:  [430, 431, 433, 436, 482, 528, 531, 544, 545],
    4:  [426, 427, 428, 576, 578, 579, 580, 581, 582]
}

# Check overlap
for cid, cres in clusters.items():
    for pid, pres in pockets.items():
        overlap = set(cres) & set(pres)
        if overlap:
            print(f"✅ Cluster {cid} overlaps with Pocket {pid} at residues: {sorted(overlap)}")


In [ ]:
!pip install ace_tools

In [ ]:
# Install dependencies
!pip install -q biopython prody
!apt install -y autodock-vina


In [ ]:
from Bio.PDB import PDBParser
import numpy as np

# ⚙️ Path to full NFATC4 protein PDB
full_pdb = "/content/drive/MyDrive/NFATC4/structure/AF-Q14934-F1-model_v4.pdb"

# 📌 Residues in Pocket 10 (from earlier result)
pocket10_residues = [434, 435, 527, 528, 529, 530, 533]

# Load structure
parser = PDBParser(QUIET=True)
structure = parser.get_structure("NFATC4", full_pdb)

# Extract atom coordinates from Pocket 10 residues
coords = []
for model in structure:
    for chain in model:
        for residue in chain:
            res_id = residue.get_id()[1]  # get residue number
            if res_id in pocket10_residues:
                for atom in residue:
                    coords.append(atom.coord)

coords = np.array(coords)

# Calculate center and box size with margin
center = coords.mean(axis=0)
size = coords.max(axis=0) - coords.min(axis=0) + 6  # Add margin of ±3Å

center_x, center_y, center_z = center
size_x, size_y, size_z = size

print(f"✅ Grid Center:  center_x = {center_x:.2f}, center_y = {center_y:.2f}, center_z = {center_z:.2f}")
print(f"✅ Grid Size:    size_x = {size_x:.2f}, size_y = {size_y:.2f}, size_z = {size_z:.2f}")
